# Octree Querying Using Python

In [138]:
import numpy as np

The purpose of this notebook is to create and test recursive functions like the octree spatial querying algorithm. Here we define a 3d space using a numpy array and put a number of objects within it at random. These objects will have bounds, which the octree will have to adequately navigate around.

To start off with, we input just the object centers in a 32 x 32 x 32 numpy cube.

In [139]:
grid_3d = np.zeros([32, 32, 32])
grid_3d.shape

(32, 32, 32)

To this space, we add 10 random "objects". These shall be point objects for the time being, but eventually we will add bounding boxes as well.

In [140]:
object_dict = {}

for i in range(1, 11):
    X = np.random.randint(0,32)
    Y = np.random.randint(0,32)
    Z = np.random.randint(0,32)

    object_dict[i] = (X, Y, Z)

    grid_3d[Z][Y][X] = i

np.count_nonzero(grid_3d)

np.int64(10)

Now we define our octree algorithms. First, we need to find the locations of these objects in space and assign them to the lowest bounds of our octree.

In [141]:
max_depth = 4

In [142]:
def make_octree(x_l, x_r, y_t, y_b, z_f, z_b, depth, grid):

    octree = {}

    if depth == 0:
        x_l = int(x_l)
        x_r = int(x_r)
        y_t = int(y_t)
        y_b = int(y_b)
        z_f = int(z_f)
        z_b = int(z_b)
        octree['bounds'] = (x_l, x_r, y_t, y_b, z_f, z_b)

        lowest_bounds = grid[z_f:z_b, y_b:y_t, x_l:x_r]
        octree['object_indices'] = list(lowest_bounds[lowest_bounds != 0])

        return octree

    octree['bounds'] = (x_l, x_r, y_t, y_b, z_f, z_b)
    octree[0] = make_octree(x_l, x_r - (x_r - x_l)/2, y_t - (y_t - y_b)/2, y_b, z_f, z_b - (z_b - z_f)/2, depth-1, grid)
    octree[1] = make_octree(x_r - (x_r - x_l)/2, x_r, y_t - (y_t - y_b)/2, y_b, z_f, z_b - (z_b - z_f)/2, depth-1, grid)
    octree[2] = make_octree(x_l, x_r - (x_r - x_l)/2, y_t, y_t - (y_t - y_b)/2, z_f, z_b - (z_b - z_f)/2, depth-1, grid)
    octree[3] = make_octree(x_r - (x_r - x_l)/2, x_r, y_t, y_t - (y_t - y_b)/2, z_f, z_b - (z_b - z_f)/2, depth-1, grid)
    octree[4] = make_octree(x_l, x_r - (x_r - x_l)/2, y_t - (y_t - y_b)/2, y_b, z_b - (z_b - z_f)/2, z_b, depth-1, grid)
    octree[5] = make_octree(x_r - (x_r - x_l)/2, x_r, y_t - (y_t - y_b)/2, y_b, z_b - (z_b - z_f)/2, z_b, depth-1, grid)
    octree[6] = make_octree(x_l, x_r - (x_r - x_l)/2, y_t, y_t - (y_t - y_b)/2, z_b - (z_b - z_f)/2, z_b, depth-1, grid)
    octree[7] = make_octree(x_r - (x_r - x_l)/2, x_r, y_t, y_t - (y_t - y_b)/2, z_b - (z_b - z_f)/2, z_b, depth-1, grid)

    return octree

Let's run a test to see what the octree looks like.

In [143]:
test_ot = make_octree(0,32, 32, 0, 0, 32, max_depth, grid=grid_3d)
test_ot

{'bounds': (0, 32, 32, 0, 0, 32),
 0: {'bounds': (0, 16.0, 16.0, 0, 0, 16.0),
  0: {'bounds': (0, 8.0, 8.0, 0, 0, 8.0),
   0: {'bounds': (0, 4.0, 4.0, 0, 0, 4.0),
    0: {'bounds': (0, 2, 2, 0, 0, 2), 'object_indices': []},
    1: {'bounds': (2, 4, 2, 0, 0, 2), 'object_indices': []},
    2: {'bounds': (0, 2, 4, 2, 0, 2), 'object_indices': []},
    3: {'bounds': (2, 4, 4, 2, 0, 2), 'object_indices': []},
    4: {'bounds': (0, 2, 2, 0, 2, 4), 'object_indices': []},
    5: {'bounds': (2, 4, 2, 0, 2, 4), 'object_indices': []},
    6: {'bounds': (0, 2, 4, 2, 2, 4), 'object_indices': []},
    7: {'bounds': (2, 4, 4, 2, 2, 4), 'object_indices': []}},
   1: {'bounds': (4.0, 8.0, 4.0, 0, 0, 4.0),
    0: {'bounds': (4, 6, 2, 0, 0, 2), 'object_indices': []},
    1: {'bounds': (6, 8, 2, 0, 0, 2), 'object_indices': []},
    2: {'bounds': (4, 6, 4, 2, 0, 2), 'object_indices': []},
    3: {'bounds': (6, 8, 4, 2, 0, 2), 'object_indices': []},
    4: {'bounds': (4, 6, 2, 0, 2, 4), 'object_indices': []}

Now lets create an intersection function.

In [144]:
def is_intersecting_3d(pos: tuple, coords: tuple, threshold=4):
    """
    Docstring for is_intersecting
    
    :param pos: Description
    :type pos: tuple
    :param input_tree: Description
    :param threshold: Description
    """
    camera_x, camera_y, camera_z = pos
    x_l, x_r, y_t, y_b, z_f, z_b = coords

    # Find where the camera is with reference to the box
    # find the X, Y and Z coordinate of the nearest side of the box
    closest_x = max(x_l, min(camera_x, x_r))
    closest_y = max(y_b, min(camera_y, y_t))
    closest_z = max(z_f, min(camera_z, z_b))

    # find the length of the line from camera center to box edge
    curr_dist = (closest_x - camera_x)**2 + (closest_y - camera_y)**2 + (closest_z - camera_z)**2
    
    if curr_dist < threshold**2:
        return True
    else:
        return False

We run a test to see if this function is working as expected. We should see a False here.

In [145]:
test_pos = (0,0,0)
test_coords = (30, 32, 32, 30, 30, 32)

is_intersecting_3d(test_pos, test_coords, threshold=5)

False

And we should see a True here.

In [146]:
test_pos = (28,28,28)
test_coords = (30, 32, 32, 30, 30, 32)

is_intersecting_3d(test_pos, test_coords, threshold=5)

True

Great. Now let's create our main search query function. This function will take in a camera position, run an intersection test at the various levels of the octree, and finally return the object indices of the octree levels which intersect with the search radius.

In [147]:
def octree_search(ot, camera_pos, radius, objects_lst, depth=max_depth):
    if (depth == 0):
        if (len(ot['object_indices']) != 0):
            objects_lst.append(ot['object_indices'])
            return
        
        return

    if is_intersecting_3d(camera_pos, ot['bounds'], radius):
        for i in range(8):
            octree_search(ot[i], camera_pos, radius, objects_lst, depth-1)

    return objects_lst

Let's see if this works. We call the octree_search function on our predefined octree.

In [148]:
index_lst = []
search_radius = 5

results = octree_search(test_ot, test_pos, search_radius, index_lst)
results

[[np.float64(7.0)]]

Let's try with different positions.

In [149]:
input_pos = [(16,16,16), (24,24,24), (24,24,8), (24,8,24), (8,24,24), (8,8,24), (8,24,8), (24,8,8), (8,8,8)]

for j in input_pos:
    index_lst = []
    results = octree_search(test_ot, j, search_radius, index_lst)
    print(results)

[]
[]
[[np.float64(8.0)]]
[[np.float64(3.0)]]
[]
[[np.float64(1.0)]]
[[np.float64(2.0)]]
[[np.float64(4.0)]]
[[np.float64(6.0)]]


We get some results. Let's vet these to make sure everything is working as expected. An important note is that the octree will return certain objects even if their distance is slightly greater than our threshold. This is becuase, the intersection test currently only works on the octree level and sesarch sphere. If the lowest level of our octree intersects with our cube, the function will return the indices of all objects in that cube. 

Note that an object may be outside of our search radius, but still within the bounds of a cube in the octree. In this case, we will get a "false postivie", but the goal is to streamline these computations, so a little bit of overestimation is fine here. A final step will be to calculate the distance between our camera and all queried objects to ensure that the distance calculations are accurate.

In [150]:
object_coords = object_dict[1]
print(object_coords)
X, Y, Z = object_coords
camera_X, camera_Y, camera_Z = (24,24,24)

# Add a tolerance to account for the case when an octree level is barely touching the search radius.
# Here 2 corresponds to the dimensions of the lowest level of the octree
tolerance = ((2**2 + 2**2 + 2**2))**(0.5)


print((X - camera_X)**2 + (Y - camera_Y)**2 + (Z - camera_Z)**2)
print((search_radius + tolerance)**2)


(X - camera_X)**2 + (Y - camera_Y)**2 + (Z - camera_Z)**2 <= (search_radius + tolerance)**2

(15, 9, 23)
307
71.64101615137753


False

Our octree querying function is working as expected. Let's scale up.

In [151]:
grid_3d = np.zeros([512, 512, 512])

object_dict = {}

for i in range(1, 20000):
    X = np.random.randint(50,462) # Add some padding to the end of the object placement
    Y = np.random.randint(50,462)
    Z = np.random.randint(50,462)

    object_dict[i] = (X, Y, Z)

    grid_3d[Z][Y][X] = i

max_depth = 7

input_pos = [(256,256,256), (384,384,384), (384,384,128), (384,128,384), (128,384,384), (128,128,384), (128,384,128), (384,128,128), (128,128,128)]

scaled_ot = make_octree(0, 512, 512, 0, 0, 512, max_depth, grid_3d)

In [152]:
for j in input_pos:
    index_lst = []
    results = octree_search(scaled_ot, j, search_radius, index_lst, depth=max_depth)
    print(results)

[]
[]
[[np.float64(2970.0)], [np.float64(13359.0)]]
[[np.float64(11898.0)]]
[]
[]
[[np.float64(9229.0)], [np.float64(4649.0)]]
[[np.float64(5899.0)]]
[]


One final step is to wrap this function in another that actually tests for distance among our returned objects. We also include a funtion that checks for the distance calcs.

In [153]:
def check_dist(camera_pos, obj_pos, radius):
    camera_X, camera_Y, camera_Z = camera_pos
    object_X, object_Y, object_Z = obj_pos

    euc_dist = (object_X - camera_X)**2 + (object_Y - camera_Y)**2 + (object_Z - camera_Z)**2

    return (euc_dist <= (radius)**2)


def octree_distance(ot, camera_pos, radius, depth):
    index_lst = []

    results = octree_search(ot, camera_pos, radius, index_lst, depth)
    
    # unnest the list
    results = [int(x) for sublist in results for x in sublist]

    # check for distance calcs
    objs_in_radius = [x for x in results if check_dist(camera_pos, object_dict[x], radius)]

    return objs_in_radius

Let's run these new functions on our predefined data.

In [154]:
for j in input_pos:
    object_lst = octree_distance(scaled_ot, j, search_radius, depth=max_depth)
    print(object_lst)

[]
[]
[]
[]
[]
[]
[9229]
[5899]
[]


Looks good. Let's create a naive search function as well.

In [155]:
def naive_search(grid, camera_pos, radius):
    search_size = list(grid[grid != 0])

    objs_in_radius = [int(x) for x in search_size if check_dist(camera_pos, object_dict[x], radius)]

    return objs_in_radius

Let's test this new function on our predefined data. We should see the same results as the octree code.

In [156]:
for j in input_pos:
    object_lst = naive_search(grid_3d, j, search_radius)
    print(object_lst)

[]
[]
[]
[]
[]
[]
[9229]
[5899]
[]


Perfect we see similar results. Now its time to time these functions and see exactly how much computatational savings we get. An important consideration here is that we wont be including the time to create our initial octree. This shall be considered as an upfront time cost, and only needs to be done once. We will compare apples to apples, by measuring performance of our functions purely in terms of querying 3D space. 

In [157]:
%timeit octree_distance(scaled_ot, (256,256,256), search_radius, depth=max_depth)

301 μs ± 4.48 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [158]:
%timeit naive_search(grid_3d, (256,256,256), search_radius)

122 ms ± 2.73 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


We see incredible time savings. Our octree querying function averages 301 microseconds for query time, while our naive search averages 119ms. The results should be even more stark as we increase the number of objects in our scene. Here we shall make our grid more dense by doubling the number of objects within it.

In [159]:
grid_3d = np.zeros([512, 512, 512])

object_dict = {}

for i in range(1, 40000):
    X = np.random.randint(50,462) # Add some padding to the end of the object placement
    Y = np.random.randint(50,462)
    Z = np.random.randint(50,462)

    object_dict[i] = (X, Y, Z)

    grid_3d[Z][Y][X] = i

max_depth = 7

input_pos = [(256,256,256), (384,384,384), (384,384,128), (384,128,384), (128,384,384), (128,128,384), (128,384,128), (384,128,128), (128,128,128)]

scaled_ot = make_octree(0, 512, 512, 0, 0, 512, max_depth, grid_3d)

In [160]:
np.count_nonzero(grid_3d)

np.int64(39987)

First lets confirm that our functions return the same results.

In [161]:
for j in input_pos:
    object_lst = octree_distance(scaled_ot, j, search_radius, depth=max_depth)
    print(object_lst)

[]
[11831]
[]
[16100]
[]
[]
[]
[17056]
[22377]


In [162]:
for j in input_pos:
    object_lst = naive_search(grid_3d, j, search_radius)
    print(object_lst)

[]
[11831]
[]
[16100]
[]
[]
[]
[17056]
[22377]


Looks good. Now, calling our timeit functions again, here is what we observe.

In [163]:
%timeit octree_distance(scaled_ot, (256,256,256), search_radius, depth=max_depth)
%timeit naive_search(grid_3d, (256,256,256), search_radius)

303 μs ± 7.66 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
132 ms ± 2.85 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


Octree querying has stayed roughly the same while the naive search has increased slightly. This is to be expected.